# Smooth Vocal & Instrumental Isolation Ensemble\n\nThis Google Colab notebook separates an audio file into **vocals** and **instrumental** stems by combining several strong open-source separation models into one smoother ensemble result.\n\nThe workflow uses:\n- **Demucs HTDemucs fine-tuned** for high-quality time/frequency separation.\n- **Demucs HTDemucs 6-source** for an alternate vocal estimate.\n- **Demucs MDX extra** for another independent separation pass.\n- Loudness matching, phase-safe averaging, residual reconstruction, and optional smoothing to make the final result less harsh.\n\n> Runtime tip: In Colab, choose **Runtime → Change runtime type → GPU** before running.

## How to use this notebook if you do not know Python

You do **not** need to install Python, write code, or set up a project. Google Colab runs everything in your browser.

### One-time setup
1. Open this `.ipynb` file in Google Colab. If you are viewing it on GitHub, click **Open in Colab** or upload the notebook at <https://colab.research.google.com/>.
2. In the Colab menu, click **Runtime → Change runtime type**.
3. Set **Hardware accelerator** to **GPU**, then click **Save**. This makes the isolation much faster.

### Run it
1. Click the first code cell named **Install dependencies**.
2. Press the round **play** button on the left side of that cell. Wait until it finishes. It can take a few minutes.
3. Run each code cell from top to bottom. You can also use **Runtime → Run all**.
4. When the **Upload audio file** cell opens a file picker, choose your song/audio file from your computer. WAV, MP3, FLAC, M4A, and many other formats should work.
5. Wait while the notebook runs the separation models. The first run is slower because it downloads model files.
6. At the end, Colab previews and downloads two files:
   - `smooth_ensemble_vocals.wav` — mostly vocals/voice.
   - `smooth_ensemble_instrumental.wav` — mostly instrumental/background.

### If something looks scary
- A warning line is not always a failure. If at least one model completes, the notebook can still create output.
- If Colab disconnects, reconnect and run the notebook again from the top.
- If your audio is very long, try a shorter file first. Songs around 3–5 minutes are a good first test.
- If you see an out-of-memory error, switch to a better GPU runtime if available or remove one model from `MODEL_NAMES` in the setup cell.

### What you can safely edit
If you understand other programming languages, the safest variables to tweak are:
- `MODEL_NAMES` — remove a model for faster processing.
- `base_weights` — change how much each model contributes to the final vocal.
- `ensemble_instrumental` blend numbers — increase residual weight to reduce vocal bleed.


## If the notebook looks like text with `\n` everywhere

That means you are looking at the raw `.ipynb` file format instead of running it inside Google Colab. An `.ipynb` notebook is saved as JSON, so raw viewers show quoted code lines and `\n` newline markers.

Use `vocal_instrumental_isolation_colab.ipynb` with Colab's **File → Upload notebook** button. Do **not** upload `vocal_instrumental_isolation_colab.py` with that button; Colab will report `Unexpected token '#'` because `.py` is normal Python text, not notebook JSON.


In [ ]:
#@title Install dependencies\n!apt-get -qq update\n!apt-get -qq install -y ffmpeg\n!pip -q install -U demucs soundfile librosa pyloudnorm pedalboard ipywidgets\n

In [ ]:
#@title Imports and setup\nfrom __future__ import annotations\n\nimport os\nimport shutil\nimport subprocess\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport librosa\nimport numpy as np\nimport pyloudnorm as pyln\nimport soundfile as sf\nfrom google.colab import files\nfrom IPython.display import Audio, display\nfrom pedalboard import HighpassFilter, LowpassFilter, Pedalboard\n\nWORK_DIR = Path('/content/isolation_workspace')\nINPUT_DIR = WORK_DIR / 'input'\nOUTPUT_DIR = WORK_DIR / 'output'\nSEPARATED_DIR = WORK_DIR / 'separated'\nfor folder in (INPUT_DIR, OUTPUT_DIR, SEPARATED_DIR):\n    folder.mkdir(parents=True, exist_ok=True)\n\nSAMPLE_RATE = 44100\nMODEL_NAMES = [\n    'htdemucs_ft',      # high-quality fine-tuned Demucs model\n    'htdemucs_6s',      # alternate 6-source model, useful vocal estimate\n    'mdx_extra',        # MDX-style Demucs model for ensemble diversity\n]\nprint('Workspace:', WORK_DIR)\n

In [ ]:
#@title Upload audio file\nuploaded = files.upload()\nif not uploaded:\n    raise RuntimeError('Upload one audio file first.')\n\ninput_name = next(iter(uploaded))\nraw_input_path = INPUT_DIR / input_name\nshutil.move(input_name, raw_input_path)\n\n# Convert everything to a clean 44.1 kHz stereo WAV so each model receives identical input.\ninput_wav = INPUT_DIR / 'source_44100_stereo.wav'\nsubprocess.run([\n    'ffmpeg', '-y', '-i', str(raw_input_path),\n    '-ar', str(SAMPLE_RATE), '-ac', '2', '-c:a', 'pcm_s16le', str(input_wav)\n], check=True)\nprint('Prepared input:', input_wav)\ndisplay(Audio(str(input_wav)))\n

In [ ]:
#@title Run multiple separation models\ndef run_demucs(model_name: str, audio_path: Path) -> Path:\n    model_out = SEPARATED_DIR / model_name\n    model_out.mkdir(parents=True, exist_ok=True)\n    cmd = [\n        'python', '-m', 'demucs.separate',\n        '--two-stems', 'vocals',\n        '-n', model_name,\n        '-o', str(model_out),\n        '--filename', '{track}/{stem}.{ext}',\n        str(audio_path),\n    ]\n    print('Running:', ' '.join(cmd))\n    subprocess.run(cmd, check=True)\n    return model_out / audio_path.stem\n\nmodel_result_dirs = {}\nfor model in MODEL_NAMES:\n    try:\n        model_result_dirs[model] = run_demucs(model, input_wav)\n    except subprocess.CalledProcessError as exc:\n        print(f'WARNING: {model} failed and will be skipped: {exc}')\n\nif not model_result_dirs:\n    raise RuntimeError('No separation model completed successfully.')\n\nmodel_result_dirs\n

In [ ]:
#@title Ensemble helpers\ndef read_audio(path: Path, sr: int = SAMPLE_RATE) -> np.ndarray:\n    audio, _ = librosa.load(path, sr=sr, mono=False)\n    if audio.ndim == 1:\n        audio = np.vstack([audio, audio])\n    return audio.T.astype(np.float32)\n\ndef match_length(audio: np.ndarray, length: int) -> np.ndarray:\n    if len(audio) > length:\n        return audio[:length]\n    if len(audio) < length:\n        return np.pad(audio, ((0, length - len(audio)), (0, 0)))\n    return audio\n\ndef rms(audio: np.ndarray) -> float:\n    return float(np.sqrt(np.mean(np.square(audio)) + 1e-12))\n\ndef loudness_normalize_to(reference: np.ndarray, candidate: np.ndarray, sr: int = SAMPLE_RATE) -> np.ndarray:\n    # Match RMS first; LUFS can be unstable for near-silent stems.\n    ref_rms = rms(reference)\n    cand_rms = rms(candidate)\n    if cand_rms < 1e-7:\n        return candidate\n    scaled = candidate * (ref_rms / cand_rms)\n\n    try:\n        meter = pyln.Meter(sr)\n        ref_lufs = meter.integrated_loudness(reference)\n        cand_lufs = meter.integrated_loudness(scaled)\n        if np.isfinite(ref_lufs) and np.isfinite(cand_lufs):\n            gain = 10 ** ((ref_lufs - cand_lufs) / 20)\n            scaled = scaled * np.clip(gain, 0.5, 2.0)\n    except Exception as err:\n        print('LUFS match skipped:', err)\n    return scaled.astype(np.float32)\n\ndef soft_clip(audio: np.ndarray, drive: float = 1.05) -> np.ndarray:\n    return np.tanh(audio * drive) / np.tanh(drive)\n\ndef smooth_stem(audio: np.ndarray, sr: int = SAMPLE_RATE) -> np.ndarray:\n    # Gentle cleanup: remove sub-rumble and very high model fizz without dulling vocals too much.\n    board = Pedalboard([HighpassFilter(cutoff_frequency_hz=25), LowpassFilter(cutoff_frequency_hz=19500)])\n    processed = board(audio.astype(np.float32), sr)\n    return np.asarray(processed, dtype=np.float32)\n\ndef weighted_average(stems: Iterable[np.ndarray], weights: Iterable[float]) -> np.ndarray:\n    stems = list(stems)\n    weights = np.asarray(list(weights), dtype=np.float32)\n    weights = weights / weights.sum()\n    stacked = np.stack(stems, axis=0)\n    return np.tensordot(weights, stacked, axes=(0, 0)).astype(np.float32)\n

In [ ]:
#@title Build the smooth ensemble result\nsource_audio = read_audio(input_wav)\ntarget_len = len(source_audio)\n\nvocal_stems = []\ninstrumental_stems = []\nused_models = []\n\nfor model, result_dir in model_result_dirs.items():\n    vocal_path = result_dir / 'vocals.wav'\n    no_vocal_path = result_dir / 'no_vocals.wav'\n    if not vocal_path.exists():\n        print(f'Skipping {model}: missing vocals.wav')\n        continue\n\n    vocals = match_length(read_audio(vocal_path), target_len)\n    # Prefer each model's no_vocals stem when present; otherwise use a residual.\n    if no_vocal_path.exists():\n        instrumental = match_length(read_audio(no_vocal_path), target_len)\n    else:\n        instrumental = source_audio - vocals\n\n    vocal_stems.append(vocals)\n    instrumental_stems.append(instrumental)\n    used_models.append(model)\n\nif not vocal_stems:\n    raise RuntimeError('No usable vocal stems were produced.')\n\n# Weight the most natural general-purpose model highest, while keeping the other models for detail.\nbase_weights = {'htdemucs_ft': 0.50, 'htdemucs_6s': 0.25, 'mdx_extra': 0.25}\nweights = [base_weights.get(model, 1.0) for model in used_models]\nreference_vocal = vocal_stems[0]\nmatched_vocals = [reference_vocal] + [loudness_normalize_to(reference_vocal, stem) for stem in vocal_stems[1:]]\n\nensemble_vocals = weighted_average(matched_vocals, weights)\nensemble_vocals = smooth_stem(ensemble_vocals)\n\n# Reconstruct instrumental as source minus final vocals for phase-safe summing, then blend with model instrumentals.\nresidual_instrumental = source_audio - ensemble_vocals\nmatched_instrumentals = [loudness_normalize_to(residual_instrumental, stem) for stem in instrumental_stems]\nmodel_instrumental_blend = weighted_average(matched_instrumentals, weights)\nensemble_instrumental = 0.70 * residual_instrumental + 0.30 * model_instrumental_blend\nensemble_instrumental = smooth_stem(ensemble_instrumental)\n\n# Prevent clipping while preserving the source level relationship.\npeak = max(np.max(np.abs(ensemble_vocals)), np.max(np.abs(ensemble_instrumental)), 1e-6)\nif peak > 0.98:\n    gain = 0.98 / peak\n    ensemble_vocals *= gain\n    ensemble_instrumental *= gain\n\nensemble_vocals = soft_clip(ensemble_vocals).astype(np.float32)\nensemble_instrumental = soft_clip(ensemble_instrumental).astype(np.float32)\n\nvocals_out = OUTPUT_DIR / 'smooth_ensemble_vocals.wav'\ninstrumental_out = OUTPUT_DIR / 'smooth_ensemble_instrumental.wav'\nsf.write(vocals_out, ensemble_vocals, SAMPLE_RATE, subtype='PCM_24')\nsf.write(instrumental_out, ensemble_instrumental, SAMPLE_RATE, subtype='PCM_24')\n\nprint('Used models:', used_models)\nprint('Saved:', vocals_out)\nprint('Saved:', instrumental_out)\n

In [ ]:
#@title Preview and download results\nprint('Vocals preview')\ndisplay(Audio(str(vocals_out)))\nprint('Instrumental preview')\ndisplay(Audio(str(instrumental_out)))\n\nfiles.download(str(vocals_out))\nfiles.download(str(instrumental_out))\n

## Tuning notes\n\n- If vocals sound too wet or phasey, reduce `mdx_extra` weight and increase `htdemucs_ft`.\n- If instrumental has too much vocal bleed, change `ensemble_instrumental = 0.70 * residual_instrumental + 0.30 * model_instrumental_blend` to use more residual, such as `0.85 / 0.15`.\n- If output sounds dull, raise the low-pass cutoff in `smooth_stem` from `19500` to `20500`.\n- For faster runs, remove one model from `MODEL_NAMES`; for smoother results, keep all three.